# CRA → Vite Migration Tool

Convert old React apps (CRA) to Vite using LLMs.  
Preview refactored configs, migrate safely, and benchmark build times.  

**Features:**
- Converts CRA projects to Vite
- Uses Hugging Face LLaMA & OpenRouter for migration
- Dry-run in temporary directory
- Benchmarks build times
- Interactive Gradio UI

In [10]:
import os
import shutil
import tempfile
import subprocess
import platform
import time
from pathlib import Path
import json
import gradio as gr

system_info = {
    "os": platform.system(),
    "arch": platform.machine(),
    "cpu_cores": os.cpu_count() or 4
}
print("System info:", system_info)

System info: {'os': 'Darwin', 'arch': 'arm64', 'cpu_cores': 10}


In [16]:
from dotenv import load_dotenv

load_dotenv(override=True)

openrouter_api_key = os.getenv("OPENAI_API_KEY")
hf_token = os.getenv("HG_TOKEN")

for name, key in [("OpenRouter", openrouter_api_key), ("Hugging Face", hf_token)]:
    if key:
        print(f"{name} API Key loaded, starts with: {key[:8]}...")
    else:
        print(f"{name} API Key not set!")

OpenRouter API Key loaded, starts with: sk-or-v1...
Hugging Face API Key loaded, starts with: hf_QZMYA...


In [17]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

HF_MODEL = "meta-llama/Llama-3.1-8B"

tokenizer = AutoTokenizer.from_pretrained(HF_MODEL, use_auth_token=hf_token)
hf_model = AutoModelForCausalLM.from_pretrained(HF_MODEL, use_auth_token=hf_token)

hf_pipeline = pipeline(
    "text-generation",
    model=hf_model,
    tokenizer=tokenizer,
    device_map="auto",
    max_new_tokens=1024
)
print(f"Hugging Face model '{HF_MODEL}' loaded successfully.")

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

/Users/josedev/Documents/research/andela/llm_engineering/.venv/lib/python3.12/site-packages/transformers/models/auto/auto_factory.py:492: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Device set to use mps:0


Hugging Face model 'meta-llama/Llama-3.1-8B' loaded successfully.


In [18]:
from openai import OpenAI

openrouter_url = "https://openrouter.ai/api/v1"
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)

In [19]:
system_prompt = """
Your task is to convert a React app created with CRA into a Vite project.
Keep all behavior identical, update scripts, dependencies, configs, and imports.
Only output changed files and content.
Be concise, output valid JSON: {"files": {"path/to/file": "new content"}}
"""

def user_prompt_for(project_path: str) -> str:
    return (
        f"Convert this CRA project at path: {project_path} to Vite. "
        "Include package.json scripts and vite.config.js updates. Output JSON only."
    )

In [20]:
def messages_for(project_path: str) -> list:
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(project_path)}
    ]

In [21]:
def convert_cra_to_vite_hf(project_path: str) -> dict:
    """Use Hugging Face LLaMA pipeline to convert CRA -> Vite"""
    prompt = "\n".join([system_prompt, user_prompt_for(project_path)])
    result = hf_pipeline(prompt)[0]["generated_text"]
    try:
        files_json = json.loads(result).get("files", {})
    except Exception:
        return {"error": f"HF LLM output parsing error: {result}"}
    return files_json

def convert_cra_to_vite_openrouter(project_path: str) -> dict:
    """Use OpenRouter GPT as backup"""
    response = openrouter.chat.completions.create(
        model=OPENROUTER_GPT,
        messages=messages_for(project_path),
        timeout=180
    )
    reply = response.choices[0].message.content.strip()
    try:
        files_json = json.loads(reply).get("files", {})
    except Exception:
        return {"error": f"OpenRouter output parsing error: {reply}"}
    return files_json

In [22]:
def apply_migration(files_json: dict, project_path: str) -> str:
    """Apply converted files to a temporary directory"""
    temp_dir = tempfile.mkdtemp()
    shutil.copytree(project_path, temp_dir, dirs_exist_ok=True)

    for rel_path, content in files_json.items():
        full_path = os.path.join(temp_dir, rel_path)
        os.makedirs(os.path.dirname(full_path), exist_ok=True)
        with open(full_path, "w", encoding="utf-8") as f:
            f.write(content)

    return temp_dir

In [23]:
def benchmark_project(path: str) -> str:
    """Install dependencies and build project, return elapsed time"""
    try:
        start = time.perf_counter()
        subprocess.run(["npm", "install"], cwd=path, check=True, capture_output=True)
        subprocess.run(["npm", "run", "build"], cwd=path, check=True, capture_output=True)
        elapsed = time.perf_counter() - start
        return f"Build completed in {elapsed:.2f}s"
    except subprocess.CalledProcessError as e:
        return f"Error during build: {e.stderr.decode()}"

In [24]:
def run_migration(project_path: str, use_hf=True):
    if use_hf:
        files_json = convert_cra_to_vite_hf(project_path)
        fallback = "HF"
    else:
        files_json = convert_cra_to_vite_openrouter(project_path)
        fallback = "OpenRouter"

    if "error" in files_json:
        return f"{fallback} Error: {files_json['error']}", "", ""

    temp_dir = apply_migration(files_json, project_path)
    benchmark_msg = benchmark_project(temp_dir)
    return json.dumps(files_json, indent=2), f"Migration temp dir: {temp_dir}", benchmark_msg

In [25]:
import gradio as gr

CSS = """
:root {
  --accent: #753991;
  --card: #161a22;
  --text: #e9eef5;
}
.gradio-container { max-width: 100% !important; padding: 0 40px !important; }
.convert-btn button { background: var(--accent) !important; color: white !important; font-weight: 700; }
"""

with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title="CRA → Vite Migrator") as ui:
    gr.Markdown("## CRA → Vite Migration Tool")

    with gr.Row():
        project_path = gr.Textbox(label="CRA Project Path", placeholder="/path/to/cra/project")
        use_hf = gr.Checkbox(label="Use Hugging Face LLaMA (unchecked = OpenRouter)", value=True)

    convert_btn = gr.Button("Migrate & Benchmark", elem_classes=["convert-btn"])
    files_output = gr.Code(label="Converted Files (JSON)", language="json", lines=20)
    temp_output = gr.Textbox(label="Temporary Migration Directory")
    benchmark_output = gr.Textbox(label="Build Benchmark")

    convert_btn.click(
        fn=lambda p, hf: run_migration(p, use_hf=hf),
        inputs=[project_path, use_hf],
        outputs=[files_output, temp_output, benchmark_output]
    )

ui.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


## Future Improvements

1. **Multi-Language Support:** Add TypeScript/Next.js CRA migration.
2. **Interactive Diff Preview:** Show diffs of migrated files before applying.
3. **Incremental Migration:** Only update files with detected CRA patterns.
4. **Custom Templates:** Allow Vite template selection (React, React + TS, PWA, etc.)
5. **Caching & Optimization:** Cache LLM responses to reduce API calls.
6. **Error Handling:** Better parsing and fallback mechanism for malformed JSON outputs.
7. **Parallel Build Benchmarking:** Test multiple migration strategies for speed comparison.
8. **Offline LLaMA:** Allow local LLaMA inference without Hugging Face API.